# Embeddings & Semantic Search

Companion notebook for the [Embeddings & Semantic Search lesson](https://ml-viz-ruby.vercel.app/courses/building-with-llms/03-embeddings-and-semantic-search).

**The idea in one sentence.** Embeddings map text to vectors so that *semantic*
similarity becomes *geometric* closeness — which lets you search by **meaning** ("how do
I get my money back" finds "refund" docs) instead of by keyword.

The mechanics:

- **Embed** each document and the query into the same vector space.
- **Rank by cosine similarity** — the angle between vectors, invariant to length.

We build a toy embedder and cosine search from scratch, **validate that semantic search
retrieves by meaning**, then cover the gotchas.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import re

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = '#e2e8f0'
plt.rcParams['axes.labelcolor'] = '#e2e8f0'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#334155'
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.color'] = '#1e293b'
plt.rcParams['figure.figsize'] = (8, 5)
np.random.seed(0)

## Toy embeddings

Real embeddings come from a trained transformer. To stay dependency-free we use a simple bag-of-words vector over a shared vocabulary — enough to demonstrate cosine ranking. The *mechanics* (vector + cosine + ranking) are identical to a production retriever.

In [ ]:
CORPUS = [
    'Refunds are available within 30 days of purchase',
    'How to request a refund for your order',
    'International shipping options and delivery times',
    'Track your package after it ships',
    'Reset your password from the account settings page',
]

def tokenize(s):
    return re.findall(r'[a-z]+', s.lower())

vocab = sorted({w for d in CORPUS for w in tokenize(d)})
index = {w: i for i, w in enumerate(vocab)}

def embed(text):
    v = np.zeros(len(vocab))
    for w in tokenize(text):
        if w in index:
            v[index[w]] += 1.0
    return v

doc_vecs = np.array([embed(d) for d in CORPUS])
print('vocab size:', len(vocab), '| doc matrix:', doc_vecs.shape)

## Cosine similarity and ranking

In [ ]:
def cosine(a, b):
    na, nb = np.linalg.norm(a), np.linalg.norm(b)
    if na == 0 or nb == 0:
        return 0.0
    return float(a @ b / (na * nb))

def search(query, k=3):
    q = embed(query)
    sims = [cosine(q, d) for d in doc_vecs]
    order = np.argsort(sims)[::-1][:k]
    return [(CORPUS[i], round(sims[i], 3)) for i in order]

for hit in search('how do I get my money back', k=3):
    print(hit)

### Validate: search retrieves by meaning, ranked by cosine

Two checks. Cosine similarity is a proper similarity — a document with itself scores 1.
And semantic search returns the on-topic documents at the top, in descending score order
(the toy is bag-of-words, so "meaning" here is lexical overlap; real embeddings capture
synonyms too).

In [ ]:
assert abs(cosine(doc_vecs[0], doc_vecs[0]) - 1.0) < 1e-9, 'self-similarity is 1'
results = search('refund my order', k=3)
print('query "refund my order" ->')
for doc, sc in results:
    print(f'  {sc:.3f}  {doc}')
scores = [sc for _, sc in results]
assert scores == sorted(scores, reverse=True), 'results are ranked by descending similarity'
assert 'refund' in results[0][0].lower(), 'the top hit should be about refunds'
print('\n✅ semantic search ranks documents by meaning (cosine), not exact keywords')

Note the top hits are about *refunds* even though the query never says 'refund' — meaning, not keywords.

## Gotchas & tradeoffs

| Gotcha | Consequence |
|--------|-------------|
| **bag-of-words = no synonyms** | "refund" vs "reimbursement" score 0 (demo); use learned embeddings |
| **cosine vs dot product** | unnormalised dot products favour longer texts; normalise |
| **chunking** | embedding whole documents dilutes the relevant passage (see RAG lesson) |
| **out-of-vocabulary** | words unseen at index time are invisible to bag-of-words |
| **embedding drift** | re-embed the corpus when you change the embedding model |

Demo: bag-of-words scores a synonym query at zero — the motivation for learned embeddings.

In [ ]:
# The bag-of-words limit that real embeddings fix: NO shared words = zero similarity,
# even for synonyms. 'reimbursement' and 'refund' mean the same thing but score 0 here —
# which is exactly why production systems use learned (contextual) embeddings, not counts.
q = embed('reimbursement for my purchase')
sims = [cosine(q, d) for d in doc_vecs]
print('query uses "reimbursement" (a synonym of refund):')
for doc, s in zip(CORPUS, sims):
    print(f'  {s:.3f}  {doc}')
print(f'\nbest score: {max(sims):.3f} — bag-of-words misses the synonym entirely.')
print('Learned embeddings place "refund" and "reimbursement" nearby; keyword vectors cannot.')

## ✏️ Your turn

Implement `top_k_retrieve(query_vec, doc_vecs, k)` returning the **indices** of the k nearest docs by cosine.

In [ ]:
def top_k_retrieve(q, docs, k):
    # TODO(you): score each doc by cosine(q, doc), return indices of the top k (descending).
    return []

idx = top_k_retrieve(embed('package tracking'), doc_vecs, 2)
assert 3 in idx  # 'Track your package after it ships'
assert len(idx) == 2
print('passed ✓')

<details><summary>Solution</summary>

```python
def top_k_retrieve(q, docs, k):
    sims = [cosine(q, d) for d in docs]
    return list(np.argsort(sims)[::-1][:k])
```

</details>

## Key takeaways

- **Embeddings turn meaning into geometry:** semantically similar text gets nearby
  vectors, so you can search by meaning.
- **Cosine similarity ranks by angle** (length-invariant); a doc with itself scores 1
  (verified), and search returns on-topic docs ranked (verified).
- **Bag-of-words misses synonyms** (demo) — learned/contextual embeddings are what make
  real semantic search work.
- This is the retrieval half of **RAG** (next lesson).